# LLM run inspection

Run one rendered scenario transition through an LLM, log the result, and inspect the submitted move.

Set at least
* `SCENARIO_NAME` to the name of a file in `outputs/scenarios`(without .json suffix!)
* `TRANSITION_INDEX` to the explicit transition from the selected scenario
* `MODEL_NAME` to the name of the model (options specified in `config/model_configs.yaml`)


The move is shown in one 2D plot per axis pair containing the move axis. A 4D move on axis 0 therefore produces the planes (0, 1), (0, 2), and (0, 3).

In [1]:
from visualization.run_figures import (
    call_prepared_llm_transition,
    display_llm_prompt,
    display_llm_response,
    display_llm_run_summary,
    finalize_llm_transition,
    plot_llm_run_move,
    prepare_llm_transition,
)

In [2]:
SCENARIO_NAME = "generator_v1"
TRANSITION_INDEX = 10
MODEL_NAME = "anthropic_claude-opus-4-8"
REASONING_EFFORT = "high"

In [3]:
# Local preparation only: load scenario, reconstruct board, and build prompts.
prepared = prepare_llm_transition(
    scenario_name=SCENARIO_NAME,
    transition_index=TRANSITION_INDEX,
    model_name=MODEL_NAME,
    reasoning_effort=REASONING_EFFORT,
)
display_llm_prompt(prepared)


### Prompt sources

- System template: [`prompts/system.txt`](../prompts/system.txt)
- User template: [`prompts/user.txt`](../prompts/user.txt)
- Model config: [`config/model_configs.yaml`](../config/model_configs.yaml)
- Scenario: [`generator_v1.json`](../outputs/scenarios/generator_v1.json)
- Backend: `litellm`

### System prompt

```text
You solve a multidimensional formal-language Scrabble benchmark.
Follow the supplied rules exactly and return only one JSON object with exactly the fields `start`, `axis`, and `sequence`.
Do not use tools or add explanations.
```

### User prompt

```text
Place exactly one contiguous sequence. Maximize the number of newly placed rack symbols.

## Move geometry
- Coordinates are zero-based vectors with 2 entries.
- `start` is the first coordinate. `axis` advances one coordinate component per symbol.
- Axis selects the board dimension along which the sequence advances: axis 0 advances coordinate index 0, axis 1 advances coordinate index 1.

## Validity rules
- The submitted sequence must be accepted by the formal language.
- Existing cells may be reused only with their existing symbol.
- Reuse at least one existing cell and place at least one new symbol.
- Only newly placed symbols consume the rack, including multiplicities.
- Do not reuse a cell whose existing word already runs along the chosen axis.
- The cell immediately before and after the submitted sequence on its axis must not continue an existing word.
- A newly placed cell must not extend an already-valid word on any perpendicular axis.
- After placement, every maximal contiguous line of length greater than one that touches the move, on every axis, must be accepted by the formal language.

## Scoring
- Each alphabet symbol has a point value, listed below. A valid move scores the sum of the point values of every symbol in its placed sequence, counting both newly placed and reused (overlapping) symbols. Higher-value symbols and longer valid sequences score more.
- An invalid move scores 0.

Letter scores:
  A: 1
  D: 1
  I: 4
  U: 3
  X: 3

Formal language:
Language ID: generator_v1_grammar
Alphabet: {A, D, I, U, X}
k: 3
Minimum word length: 3
Forbidden snippets: {A A D, A A I, A A U, A D I, A D X, A I A, A I I, A I U, A U D, A U X, A X I, A X U, D A A, D A D, D A I, D U D, D U I, D U U, D U X, D X A, D X D, D X I, I A D, I A I, I D D, I D U, I U D, I U I, I U U, I U X, I X A, I X U, U A U, U A X, U D D, U I A, U I U, U I X, U U A, U U X, X A A, X A X, X D D, X I D, X I U, X U D, X U X, X X X}
A sequence is valid iff it has the minimum length of 3 and contains no forbidden snippet.

Board configuration:
[omitted from notebook display: 48 occupied cells]

Rack:
["D", "U", "U", "X"]

```


In [4]:
timed_response = call_prepared_llm_transition(prepared)

In [5]:
context = finalize_llm_transition(prepared, timed_response)
display_llm_response(timed_response)
display_llm_run_summary(context)

sequence,OK
spatial,OK
overlap,OK
no word extension,OK
cross words,OK
rack,OK
word length,6
overlap count,2
letter score,14


In [6]:
for figure in plot_llm_run_move(context, move_source="parsed"):
    display(figure)

# Keep in Mind

Ground truth != only solution

In [7]:
for figure in plot_llm_run_move(context, move_source="ground_truth"):
    display(figure)